# 07 - Ablations: MCDM Variants (9-11)

**Purpose.** MCDM-side ablations from PROJECT_2_PLAN.md S10:

9. Equal weights vs tuned weights
10. Single-protocol benchmarks (Aave-only, Compound-only)
11. Greedy max-forecasted-APY vs TOPSIS-MCDM

Each section runs a backtest variant on the synthetic panel.

**Expected runtime.** 1-2 minutes total.


In [ ]:
# --- path preamble: make sibling packages importable ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "extras" / "fractal_pr_lending_allocation"))

print(f"ROOT = {ROOT}")


In [ ]:
# Synthetic-data fallback: mirrors forecaster.train._make_synth_df.
# Used whenever the real joined_clean.parquet is not yet on disk.
import numpy as np
import pandas as pd


def make_synth_joined(n_rows: int = 2000, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_rows, freq="h", tz="UTC")

    def util_walk(start: float) -> np.ndarray:
        u = np.empty(n_rows)
        u[0] = start
        for i in range(1, n_rows):
            u[i] = np.clip(u[i - 1] + rng.normal(0.0, 0.01), 0.05, 0.97)
        return u

    u_a = util_walk(0.55)
    u_c = util_walk(0.45)

    # Toy rate process: kink-shaped baseline plus Gaussian residual.
    r_a = 0.05 * (u_a / 0.92) * 0.90 * u_a + rng.normal(0, 0.002, n_rows)
    r_c = 0.04 * u_c + rng.normal(0, 0.002, n_rows)
    r_a = np.clip(r_a, 0.0, 0.5)
    r_c = np.clip(r_c, 0.0, 0.5)

    tvl_a = 1e8 + np.cumsum(rng.normal(0, 1e5, n_rows))
    tvl_c = 5e7 + np.cumsum(rng.normal(0, 5e4, n_rows))
    gas = np.clip(
        20 + 5 * rng.standard_normal(n_rows) + 10 * np.sin(np.arange(n_rows) / 24),
        5, 200,
    )
    eth = 3000 + np.cumsum(rng.normal(0, 5, n_rows))

    return pd.DataFrame({
        "r_aave": r_a, "r_compound": r_c,
        "u_aave": u_a, "u_compound": u_c,
        "tvl_aave": tvl_a, "tvl_compound": tvl_c,
        "gas_gwei": gas, "eth_usd": eth,
    }, index=idx)


def load_joined(path: str = "data/cached/joined_clean.parquet") -> tuple[pd.DataFrame, bool]:
    """Try the real cached panel; fall back to synthetic on FileNotFoundError."""
    full = ROOT / path
    try:
        df = pd.read_parquet(full)
        print(f"[real] loaded {len(df):,} rows from {full}")
        return df, True
    except FileNotFoundError:
        print(f"[synth] {full} not found - generating synthetic panel")
        return make_synth_joined(), False


df, is_real = load_joined()
df.head()


In [ ]:
from backtest.observations_builder import build_all
_, (_, val_obs, test_obs) = build_all(synthetic=True)
print(f'val={len(val_obs)} test={len(test_obs)}')


## Ablation 9 - Equal weights vs tuned MCDM weights

In [ ]:
from strategies.baseline_mcdm_ema import MCDMEMAStrategy, MCDMEMAParams

# Tuned defaults from AI Yield Vault paper.
tuned = MCDMEMAParams(
    INITIAL_BALANCE=1_000_000.0, DEFAULT_INITIAL_ENTITY='AAVE',
    W_APY=0.40, W_RISK=0.25, W_COST=0.20, W_STAB=0.15,
)
equal = MCDMEMAParams(
    INITIAL_BALANCE=1_000_000.0, DEFAULT_INITIAL_ENTITY='AAVE',
    W_APY=0.25, W_RISK=0.25, W_COST=0.25, W_STAB=0.25,
)

for name, p in [('tuned', tuned), ('equal', equal)]:
    strat = MCDMEMAStrategy(debug=False, params=p)
    r = strat.run(test_obs)
    m = r.get_default_metrics()
    print(f'{name}: APY={getattr(m, "apy", None)}  '
          f'Sharpe={getattr(m, "sharpe", None)}')


## Ablation 10 - Single-protocol benchmarks

In [ ]:
from strategies.baseline_buyhold import BuyAndHoldAaveStrategy
from base_lending_allocation import BaseLendingAllocationParams

# Aave-only: standard buy-and-hold strategy.
aave_only = BaseLendingAllocationParams(
    INITIAL_BALANCE=1_000_000.0, DEFAULT_INITIAL_ENTITY='AAVE',
)
r_aave = BuyAndHoldAaveStrategy(debug=False, params=aave_only).run(test_obs)
print('Aave-only:', r_aave.get_default_metrics())

# Compound-only: same strategy but starting on COMPOUND.
comp_only = BaseLendingAllocationParams(
    INITIAL_BALANCE=1_000_000.0, DEFAULT_INITIAL_ENTITY='COMPOUND',
)
r_comp = BuyAndHoldAaveStrategy(debug=False, params=comp_only).run(test_obs)
print('Compound-only:', r_comp.get_default_metrics())


## Ablation 11 - Greedy max-APY vs TOPSIS-MCDM

In [ ]:
from strategies.baseline_apy_greedy import APYGreedyStrategy, APYGreedyParams

params_greedy = APYGreedyParams(
    INITIAL_BALANCE=1_000_000.0, DEFAULT_INITIAL_ENTITY='AAVE',
)
r_greedy = APYGreedyStrategy(debug=False, params=params_greedy).run(test_obs)
r_mcdm   = MCDMEMAStrategy(debug=False, params=tuned).run(test_obs)

for name, res in [('greedy', r_greedy), ('mcdm', r_mcdm)]:
    m = res.get_default_metrics()
    print(f'{name}: APY={getattr(m, "apy", None)} '
          f'turnover-rebalances vs total observations '
          f'{len(res.to_dataframe())}')


## Next steps

- Replace the greedy-vs-MCDM with the **forecasted-rate** versions once   Predictive MCDM strategy is wired (Week 2 Day Fri 29).
- Plot the same equity overlay as in `04_main_backtest.ipynb` restricted   to the three MCDM variants.

Relevant plan section: **PROJECT_2_PLAN.md S10, Ablations 9-11.**
